# Scaling: Distributed Training, FSDP, DeepSpeed Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Simulate Data Parallelism

Split a batch across simulated GPUs. Each GPU computes a forward pass on its shard. Average the "gradients" (we simulate them as the loss values).

In [ ]:
```python

import numpy as np

def simulate_data_parallelism(data, num_gpus, model_fn):

    batch_size = len(data)

    shard_size = batch_size // num_gpus

    remainder = batch_size % num_gpus

    gpu_losses = []

    gpu_gradients = []

    offset = 0

    for gpu_id in range(num_gpus):

        extra = 1 if gpu_id < remainder else 0

        shard = data[offset:offset + shard_size + extra]

        offset += shard_size + extra

        loss, grad = model_fn(shard)

        gpu_losses.append(loss)

        gpu_gradients.append(grad)

    avg_loss = np.mean(gpu_losses)

    avg_gradient = np.mean(gpu_gradients, axis=0)

    return avg_loss, avg_gradient

In [ ]:
```

The all-reduce operation (averaging gradients) is the only communication in data parallelism. In practice, this uses the NCCL library on NVIDIA GPUs, which implements ring all-reduce: each GPU sends 1/N of its gradients to its neighbor, receives 1/N from the other neighbor, and after N-1 steps every GPU has the complete average. Total communication volume: 2 x gradient_size x (N-1)/N, approaching 2x the gradient size for large N.

### Step 2: Simulate Tensor Parallelism

Split a weight matrix across GPUs. Each GPU computes a partial matrix multiplication. Combine the results.

In [ ]:
```python

def simulate_tensor_parallelism(input_data, weight_matrix, num_gpus):

    d_in, d_out = weight_matrix.shape

    assert d_out % num_gpus == 0, f"d_out {d_out} not divisible by num_gpus {num_gpus}"

    shard_size = d_out // num_gpus

    partial_results = []

    for gpu_id in range(num_gpus):

        start = gpu_id * shard_size

        end = start + shard_size

        weight_shard = weight_matrix[:, start:end]

        partial = input_data @ weight_shard

        partial_results.append(partial)

    full_output = np.concatenate(partial_results, axis=-1)

    direct_output = input_data @ weight_matrix

    error = np.abs(full_output - direct_output).max()

    return full_output, error

In [ ]:
```

The error should be exactly zero (or machine epsilon). Tensor parallelism is mathematically exact -- it produces the same result as computing the full matmul on one GPU. The split is along the output dimension, so each GPU produces a different chunk of columns, and concatenation reconstructs the full result.

For column-parallel linear layers (splitting the output dimension), you concatenate. For row-parallel (splitting the input dimension), you sum. In a transformer FFN, the first linear (expand) uses column-parallel and the second linear (contract) uses row-parallel. This avoids an all-reduce between the two layers.

### Step 3: Simulate Pipeline Parallelism

Split a model's layers across virtual GPUs. Show the bubble problem where early stages sit idle while later stages compute.

In [ ]:
```python

def simulate_pipeline_parallelism(num_layers, num_stages, num_microbatches):

    layers_per_stage = num_layers // num_stages

    timeline = {}

    clock = 0

    for mb in range(num_microbatches):

        for stage in range(num_stages):

            start_time = max(

                timeline.get((stage, mb - 1, "fwd"), (0, 0))[1] if mb > 0 else 0,

                timeline.get((stage - 1, mb, "fwd"), (0, 0))[1] if stage > 0 else 0,

            )

            end_time = start_time + layers_per_stage

            timeline[(stage, mb, "fwd")] = (start_time, end_time)

    last_fwd_end = max(v[1] for v in timeline.values())

    for mb in range(num_microbatches - 1, -1, -1):

        for stage in range(num_stages - 1, -1, -1):

            deps = [last_fwd_end]

            if mb < num_microbatches - 1 and (stage, mb + 1, "bwd") in timeline:

                deps.append(timeline[(stage, mb + 1, "bwd")][1])

            if stage < num_stages - 1 and (stage + 1, mb, "bwd") in timeline:

                deps.append(timeline[(stage + 1, mb, "bwd")][1])

            start_time = max(deps)

            end_time = start_time + layers_per_stage

            timeline[(stage, mb, "bwd")] = (start_time, end_time)

    total_time = max(v[1] for v in timeline.values())

    compute_time = num_microbatches * num_stages * layers_per_stage * 2

    bubble_fraction = 1.0 - compute_time / (total_time * num_stages)

    return timeline, total_time, bubble_fraction

In [ ]:
```

With 4 stages and 1 micro-batch, the bubble fraction is 75% -- three out of four GPUs idle at any time. With 16 micro-batches, it drops to about 19%. The cost of eliminating bubbles is memory: you must store activations for all in-flight micro-batches simultaneously.

### Step 4: Memory Calculator

Compute the exact memory requirements for training any model size.

In [ ]:
```python

def memory_calculator(

    params_billions,

    precision_bytes=2,

    optimizer="adam",

    num_gpus=1,

    sharding="none",

    sequence_length=2048,

    batch_size_per_gpu=1,

    hidden_dim=None,

    num_layers=None,

):

    params = params_billions * 1e9

    weight_memory = params * precision_bytes

    if optimizer == "adam":

        optimizer_memory = params * 4 * 2

    elif optimizer == "sgd":

        optimizer_memory = params * 4

    else:

        optimizer_memory = 0

    gradient_memory = params * precision_bytes

    total_no_activation = weight_memory + optimizer_memory + gradient_memory

    if hidden_dim and num_layers:

        activation_per_layer = (

            sequence_length * batch_size_per_gpu * hidden_dim * precision_bytes * 4

        )

        activation_memory = activation_per_layer * num_layers

    else:

        activation_memory = params * precision_bytes * 0.5

    if sharding == "fsdp" or sharding == "zero3":

        weight_memory /= num_gpus

        optimizer_memory /= num_gpus

        gradient_memory /= num_gpus

    elif sharding == "zero2":

        optimizer_memory /= num_gpus

        gradient_memory /= num_gpus

    elif sharding == "zero1":

        optimizer_memory /= num_gpus

    per_gpu_total = weight_memory + optimizer_memory + gradient_memory + activation_memory

    return {

        "params_billions": params_billions,

        "weights_gb": weight_memory / 1e9,

        "optimizer_gb": optimizer_memory / 1e9,

        "gradients_gb": gradient_memory / 1e9,

        "activations_gb": activation_memory / 1e9,

        "per_gpu_total_gb": per_gpu_total / 1e9,

        "total_across_gpus_gb": per_gpu_total * num_gpus / 1e9,

        "fits_on_80gb": per_gpu_total / 1e9 <= 80,

        "num_gpus": num_gpus,

        "sharding": sharding,

    }

In [ ]:
```

This calculator answers the question every ML engineer asks: "How many GPUs do I need?" Feed it the model size and see whether it fits. Adjust sharding strategy until the per-GPU total drops below 80GB.

### Step 5: Mixed Precision Simulation

Compare memory usage between FP32, FP16, and mixed precision training.

In [ ]:
```python

def mixed_precision_comparison(params_billions):

    params = params_billions * 1e9

    fp32_weights = params * 4

    fp32_optimizer = params * 4 * 2

    fp32_gradients = params * 4

    fp32_total = fp32_weights + fp32_optimizer + fp32_gradients

    fp16_weights = params * 2

    fp16_master = params * 4

    fp16_optimizer = params * 4 * 2

    fp16_gradients = params * 2

    fp16_total = fp16_weights + fp16_master + fp16_optimizer + fp16_gradients

    mixed_weights = params * 2

    mixed_optimizer = params * 4 * 2

    mixed_gradients = params * 2

    mixed_total = mixed_weights + mixed_optimizer + mixed_gradients

    return {

        "fp32_total_gb": fp32_total / 1e9,

        "fp16_with_master_gb": fp16_total / 1e9,

        "mixed_bf16_gb": mixed_total / 1e9,

        "savings_vs_fp32": 1 - mixed_total / fp32_total,

    }

In [ ]:
```

The biggest surprise for most people: mixed precision does not halve the memory. The optimizer states (Adam's m and v) stay in FP32 regardless of precision. For a 7B model, FP32 training uses 112GB. Mixed precision uses 84GB. That is a 25% reduction, not 50%. The optimizer dominates.

## Exercises

In [ ]:
1. Modify the memory calculator to include activation checkpointing. With checkpointing, only store activations at every K-th layer (typical K=1, meaning recompute all). Show the memory-compute tradeoff: how much memory does checkpointing save, and how much does it slow down training (roughly 33% more compute for full checkpointing)?

2. Extend the pipeline parallelism simulation to implement the 1F1B (one forward, one backward) schedule used by PipeDream. Compare the bubble fraction against the naive schedule for 4 stages and 8 micro-batches. The 1F1B schedule should have a smaller peak memory because it starts backward passes earlier.

3. Implement a gradient accumulation simulator. Instead of all-reducing after every micro-batch, accumulate gradients locally for K steps, then all-reduce. Show how this reduces communication by K times but produces identical final gradients (and thus identical training).

4. Build a cost estimator. Given a model size, target token count, GPU type (A100 at $2/hr, H100 at $3.50/hr), and parallelism strategy, estimate the total training cost in dollars. Validate against known costs: Llama 3 405B reportedly cost ~$100M, DeepSeek V3 cost ~$5.6M.

5. Add ZeRO-Offload to the memory calculator. Assume CPU RAM is 512GB per node and NVMe is 2TB. Show how offloading optimizer states to CPU allows a 70B model to train on 4 GPUs instead of 16, at the cost of 30-50% slower optimizer steps.